# Improved transformer implementation

The previously implemented transformer version is quite basic and not efficient. In this notebook I will try to re-implement it from scratch by:
- separating $\mathbf{K}$ and $\mathbf{Q}$, $\mathbf{V}$ into different `nn.Linear` layers.
- moving out attention into a distinct `nn.Module`.

This should improve the performance and streamline the code.

## Some details
### Layer Norm and residuals
**Key idea:** it forces a layer to learn a change to the input instead of learning a whole new mapping. It might be close to an error-state in filtering, that is easy to track because it is linear.

**Interpretation:** we can think of an output of `LayerNorm` as a correction to an input to obtain our target value. Layers refine the estimate rather than overwrite it.

### Decoder
**Key idea:** in decoder two information flows are blended inside attention.

**Interpretation:** this is achieved through linear maps which align dimensions. $Q$, $K$ and $V$ are generated as:
- $Q = X_{dec}W_Q$
- $K = X_{enc}W_K$
- $V = X_{enc}W_V$

so the queries are taken from the decoder, but keys and values from encoder. The information is then mixed in $A$ and context is obtained as $AV$.
So decoder "asks" questions of the encoder memory and extracts what it needs. Each head can be interpreted as a topic, or a set of questions/dictionaries that this head can reply to.

### Masks
- **Cross-attention** or **encoder-decoder attention:** usually only an encoder padding mask (special symbols and blank/absent tokens). 
- **Self-attention:** causal mask, used in decoder to prevent the NN from "looking forward" into the context.

Always applied before softmax in attention.

Typical shapes are:
- `[B, 1, 1, K]` - for padding, broadcast over heads and queries.
- `[1, 1, Q, K]` - for causal reasoning.
- `[B, 1, 1, S_enc]` - for cross-attn padding (encoder memory).

#### Encoder mask
Used on **self-attention** for padding.

The purpose is to prevent attending padding tokens, which carry no meaning.

Application: applied to logits ($QK^T$) as additive $- \inf$ for padded $K$ positions.

#### Decoder mask
- For masked self-attention: causal, or look-ahead mask. The purpose is to forbid future tokens during training. Applied to logits, has upper-triangular shape.
- For cross-attention: padding masks to disable attending padded source and target positions.

#### Loss masking
Padding targets should be excluded from loss so padding doesn't affect the loss function value.

### Attending
Cross-attention sublayers inside the decoder attend to encoder output. All the decoder layers attend to the same (and final!) encoder output.

## Implementation

In [ ]:
import torch
from torch import nn
import math

def create_key_padding_mask(lengths, max_len, device=None):
    """Creates padding mask from input lengths
    """
    indices = torch.arange(max_len, device=device).unsqueeze(0)
    pad_mask = indices >= lengths.unsqueeze(1)
    return pad_mask.view(-1, 1, 1, max_len)

def create_causal_mask(len_q, len_k=None, device=None):
    if len_k is None:
        len_k = len_q

    return torch.ones(len_q, len_k, dtype=torch.bool, device=device).triu(diagonal=1).view(1, 1, len_q, len_k)

def sinusoidal_pe(max_len: int, d_model: int, device=None):
    pos = torch.arange(max_len, device=device).float().unsqueeze(1)           # [L,1]
    i   = torch.arange(0, d_model, 2, device=device).float()                  # [d/2]
    div = torch.exp(-math.log(10000.0) * i / d_model)                         # [d/2]
    pe  = torch.zeros(max_len, d_model, device=device)
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe

class Attention(nn.Module):
    def forward(self, q, k, v, mask=None):
        # q, k: [B, H, S, D]
        d_k = q.size(-1)
        logits = q @ k.transpose(-2, -1) / (d_k ** 0.5) # [B, H, S, S]
        if mask is not None:
            logits = logits.masked_fill(mask, -torch.inf)
        A = torch.softmax(logits, dim=-1)
        return A @ v

class MultiheadAttention(nn.Module):
    def __init__(self, input_dim, num_heads, head_dim):
        super().__init__()

        self.num_heads = num_heads
        self.head_dim = head_dim

        self.q_linear = nn.Linear(input_dim, num_heads * head_dim)
        self.kv_linear = nn.Linear(input_dim, num_heads * head_dim * 2)

        self.out_linear = nn.Linear(num_heads * head_dim, input_dim)

        nn.init.xavier_uniform_(self.q_linear.weight)
        nn.init.xavier_uniform_(self.kv_linear.weight)
        nn.init.xavier_uniform_(self.out_linear.weight)

    def forward(self, q_input, kv_input=None, mask=None):
        batch_size, seq_length, _ = q_input.size()
        if kv_input is None:
            kv_input = q_input

        q = self.q_linear(q_input)
        kv = self.kv_linear(kv_input)

        q = q.view(batch_size, seq_length, self.num_heads, self.head_dim)
        kv = kv.view(batch_size, seq_length, self.num_heads, self.head_dim * 2)
        k, v = torch.chunk(kv, 2, -1)
        q = q.permute(0, 2, 1, 3)  # [B, H, S, D]
        k = k.permute(0, 2, 1, 3)  # [B, H, S, D]
        v = v.permute(0, 2, 1, 3)  # [B, H, S, D]

        attn = Attention()(q, k, v, mask)                  # [B, H, S, D]
        attn = attn.permute(0, 2, 1, 3).contiguous() # [B, S, H, D]
        attn = attn.view(batch_size, seq_length, self.num_heads * self.head_dim)

        y = self.out_linear(attn)
        return y
    
class EncoderBlock(nn.Module):
    def __init__(self, input_dim, num_heads, head_dim, ff_dim, dropout_rate=0.1):
        super().__init__()
        self.mha = MultiheadAttention(input_dim, num_heads, head_dim)
        self.norm_mha = nn.LayerNorm(input_dim)
        self.ff = nn.Sequential(
            nn.Linear(input_dim, ff_dim),
            nn.Dropout(dropout_rate),
            nn.ReLU(),
            nn.Linear(ff_dim, input_dim),
            nn.Dropout(dropout_rate)
        )
        self.norm_ff = nn.LayerNorm(input_dim)

    def forward(self, x, self_attn_mask=None):
        mha_out = self.mha(x, mask=self_attn_mask)
        # residual connection - layer learns a change to its input, instead of a whole new mapping
        x = self.norm_mha(x + mha_out)
        ff_out = self.ff(x)
        # once again, residual connection - neural network learns corrections to its input
        x = self.norm_ff(x + ff_out)
        return x

class Encoder(nn.Module):
    def __init__(self, input_dim, num_heads, head_dim, ff_dim, num_layers, dropout_rate=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderBlock(input_dim, num_heads, head_dim, ff_dim, dropout_rate)
            for _ in range(num_layers)
        ])

    def forward(self, x, self_attn_mask):
        for layer in self.layers:
            x = layer(x, self_attn_mask)
        return x
    

class DecoderBlock(nn.Module):
    def __init__(self, input_dim, num_heads, head_dim, ff_dim, dropout_rate=0.1):
        super().__init__()
        
        self.self_attn = MultiheadAttention(input_dim, num_heads, head_dim)
        self.self_attn_norm = nn.LayerNorm(input_dim)

        self.cross_attn = MultiheadAttention(input_dim, num_heads, head_dim)
        self.cross_attn_norm = nn.LayerNorm(input_dim)

        self.ff = nn.Sequential(
            nn.Linear(input_dim, ff_dim),
            nn.Dropout(dropout_rate),
            nn.ReLU(),
            nn.Linear(ff_dim, input_dim),
            nn.Dropout(dropout_rate)
        )
        self.norm_ff = nn.LayerNorm(input_dim)

    def forward(self, x, x_enc, self_attn_mask=None, cross_attn_mask=None):
        self_attn_out = self.self_attn(x, mask=self_attn_mask)
        x = self.self_attn_norm(x + self_attn_out)

        cross_attn_out = self.cross_attn(x, x_enc, mask=cross_attn_mask)
        x = self.cross_attn_norm(x + cross_attn_out)

        ff_out = self.ff(x)
        x = self.norm_ff(x + ff_out)

        return x
    
class Decoder(nn.Module):
    def __init__(self, input_dim, num_heads, head_dim, ff_dim, num_layers, dropout_rate=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            DecoderBlock(input_dim, num_heads, head_dim, ff_dim, dropout_rate)
            for _ in range(num_layers)
        ])

    def forward(self, x, x_enc, self_attn_mask=None, cross_attn_mask=None):
        for layer in self.layers:
            x = layer(x, x_enc, self_attn_mask, cross_attn_mask)
        return x

class TransformerSeq2Seq(nn.Module):
    def __init__(self, input_dim, num_heads, head_dim, ff_dim, num_layers, d_model, max_len=512, dropout_rate=0.1):
        super().__init__()
        self.in_proj = nn.Linear(input_dim, d_model)
        self.encoder = Encoder(d_model, num_heads, head_dim, ff_dim, num_layers, dropout_rate)
        self.decoder = Decoder(d_model, num_heads, head_dim, ff_dim, num_layers, dropout_rate)
        self.out_proj = nn.Linear(d_model, input_dim)

        self.pe = sinusoidal_pe(max_len, d_model, device=None)
        self.register_buffer('positional_encoding', pe)

    def forward(self, x, x_lengths):
        B, S, _ = x.shape
        x = self.in_proj(x) + self.pe[:S].unsqueeze(0).to(x.device)  # [B,S,d_model]

        padding_mask = create_key_padding_mask(x_lengths, S, device=x.device)
        enc_out = self.encoder(x, padding_mask)

        causal_mask = create_causal_mask(S, S, device=x.device)
        pad_q = padding_mask.transpose(-1, -2)
        self_attn_mask = padding_mask | causal_mask | pad_q
        y = self.out_proj(self.decoder(x, enc_out, self_attn_mask, padding_mask))

        return y

B, S, D_in = 3, 5, 16  # Batch size, sequence length, input dimension
H, D_head = 4, 8       # Number of heads, head dimension
D_ff = 64            # Feedforward dimension
N_layers = 6
D_m = H * D_head  # Embedding dimension

x = torch.randn(B, S, D_in)
pe = sinusoidal_pe(x.size(1), x.size(2), device=x.device)
x = x + pe[None, :, :]
x_lengths = torch.randint(0, x.size(1), (x.size(0),))

transformer = Transformer(D_in, H, D_head, D_ff, N_layers, D_m, dropout_rate=0.1)
output = transformer(x, x_lengths)
print(output.shape)  # Expected output shape: [B, S, D_in]

# eb = EncoderBlock(D_in, H, D_head, D_ff, dropout_rate=0.1)
# output = eb(x)
# print(output.shape)  # Expected output shape: [B, S, D_in]

# db = DecoderBlock(D_in, H, D_head, D_ff, dropout_rate=0.1)
# output = db(x, output)
# print(output.shape)

torch.Size([3, 5, 16])


Tomorrow:
- In our case input and output sequences can be different, that is very important, 
so S is the length of the input sequence and T is the length of the targets (labels).